# XGBoost + Optuna Hyperparameter Tuning

**Week 2, Day 5:** Train XGBoost with Optuna hyperparameter optimization.

**Objectives:**
1. Understand XGBoost and its hyperparameters
2. Use Optuna for automatic hyperparameter tuning
3. Train optimized XGBoost model
4. Compare with baseline models

**Reference:** project_guide.md Week 2 Day 5

## 1. Setup & Install Optuna

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
from datetime import datetime

# Install optuna if not installed
try:
    import optuna
except ImportError:
    print("Installing optuna...")
    %pip install optuna -q
    import optuna

# XGBoost
from xgboost import XGBClassifier

# Scikit-learn
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score, precision_recall_curve, auc
)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print(f"Optuna version: {optuna.__version__}")
print("Libraries imported successfully")

## 2. Load Data

In [ ]:
processed_dir = Path('../data/processed')

X_train = joblib.load(processed_dir / 'X_train.pkl')
X_test = joblib.load(processed_dir / 'X_test.pkl')
y_train = joblib.load(processed_dir / 'y_train.pkl')
y_test = joblib.load(processed_dir / 'y_test.pkl')

print(f"Train: {X_train.shape[0]:,} samples x {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]:,} samples x {X_test.shape[1]} features")
print(f"\nFraud rate - Train: {y_train.mean():.4%}, Test: {y_test.mean():.4%}")

## 3. Understanding XGBoost Key Concepts

### What is XGBoost?

**XGBoost** = **E**xtreme **G**radient **Boost**ing

#### How Gradient Boosting Works:

```
1. Train first tree on data
2. Calculate errors (residuals)
3. Train second tree to predict those errors
4. Combine predictions: Tree1 + learning_rate × Tree2
5. Repeat for n_estimators trees
```

#### Key Hyperparameters:

| Parameter | What it controls | Effect |
|-----------|-----------------|--------|
| `max_depth` | Tree depth (3-10) | Higher = more complex, risk of overfitting |
| `learning_rate` | Step size shrinkage (0.01-0.3) | Lower = slower learning, often better final |
| `n_estimators` | Number of trees (100-1000) | More trees = better, but diminishing returns |
| `subsample` | Fraction of data per tree (0.6-1.0) | < 1.0 prevents overfitting (stochastic) |
| `colsample_bytree` | Fraction of features per tree | < 1.0 prevents overfitting |
| `scale_pos_weight` | Weight for positive class | CRITICAL for imbalanced data! |

#### For Imbalanced Data (Fraud Detection):

**Formula for scale_pos_weight:**
```python
scale_pos_weight = (negative_cases) / (positive_cases)
                   = 284315 / 492
                   ≈ 578
```

In [ ]:
# Calculate scale_pos_weight for our data
neg_cases = (y_train == 0).sum()
pos_cases = (y_train == 1).sum()
scale_pos_weight = neg_cases / pos_cases

print(f"=== Class Imbalance ===")
print(f"Negative (legitimate): {neg_cases:,}")
print(f"Positive (fraud): {pos_cases:,}")
print(f"\nRecommended scale_pos_weight: {scale_pos_weight:.1f}")
print(f"\nThis means: Each fraud case is weighted as {scale_pos_weight:.0f} legitimate cases!")

## 4. Understanding Optuna

### What is Optuna?

**Optuna** = Automatic hyperparameter optimization framework

#### Why Optuna over Grid/Random Search?

| Method | Trials needed | How it works |
|--------|---------------|-------------|
| Grid Search | 1000s | Tries EVERY combination (exponential) |
| Random Search | 100s | Tries random combinations |
| **Optuna** | **~50** | **Learns from previous trials (Bayesian)** |

#### How Optuna Works:

```
1. Define objective function (returns metric to optimize)
2. Optuna suggests hyperparameters
3. Train model, get score
4. Optuna learns: "These params gave good score, try nearby"
5. Repeat for n_trials
```

#### Key Components:

```python
# Search space (what to tune)
trial.suggest_int('max_depth', 3, 10)        # Integer: 3 to 10
trial.suggest_float('learning_rate', 0.01, 0.3)  # Float: 0.01 to 0.3
trial.suggest_categorical('booster', ['gbtree', 'gblinear'])  # Choice

# Study (optimization process)
study = optuna.create_study(direction='maximize')  # Maximize ROC-AUC
study.optimize(objective, n_trials=50)

# Best results
best_params = study.best_params
best_score = study.best_value
```

## 5. Define Optuna Objective Function

In [ ]:
def objective(trial, X_train, y_train, n_folds=5):
    """
    Optuna objective function for XGBoost hyperparameter tuning.
    
    Args:
        trial: Optuna trial object
        X_train: Training features
        y_train: Training labels
        n_folds: Number of cross-validation folds
    
    Returns:
        Mean ROC-AUC score from cross-validation
    """
    
    # Define hyperparameter search space
    params = {
        # Tree structure
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        
        # Learning
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        
        # Regularization (prevent overfitting)
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # Regularization parameters
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        
        # Class imbalance (CRITICAL!)
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', scale_pos_weight * 0.5, scale_pos_weight * 1.5),
        
        # Other
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss',
        'use_label_encoder': False,
    }
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    xgb = XGBClassifier(**params)
    
    # Use ROC-AUC for evaluation (better for imbalanced data)
    scores = cross_val_score(xgb, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    return scores.mean()

print("Objective function defined")

## 6. Run Optuna Optimization

**Note:** Using 50 trials instead of 100 to save time. You can increase to 100 for better results.

In [ ]:
print("Starting Optuna optimization...")
print("This may take 10-20 minutes depending on your CPU.")
print(f"\nRunning {50} trials...")

start_time = time.time()

# Create study
study = optuna.create_study(
    direction='maximize',  # Maximize ROC-AUC
    study_name='xgboost_fraud_detection'
)

# Run optimization
study.optimize(
    lambda trial: objective(trial, X_train, y_train),
    n_trials=50,
    show_progress_bar=True,
    n_jobs=-1
)

elapsed_time = time.time() - start_time

print(f"\n{'='*60}")
print(f"Optimization completed in {elapsed_time/60:.1f} minutes")
print(f"{'='*60}")

## 7. Best Hyperparameters Found

In [ ]:
print("\n" + "="*60)
print("          OPTUNA OPTIMIZATION RESULTS")
print("="*60)

print(f"\nBest ROC-AUC Score: {study.best_value:.4f}")

print("\nBest Hyperparameters:")
for param, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  {param:20s}: {value:.4f}")
    else:
        print(f"  {param:20s}: {value}")

print("\n" + "="*60)

## 8. Visualize Optimization History

In [ ]:
# Plot optimization history
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

# Plot parameter importance
fig = optuna.visualization.plot_param_importances(study)
fig.show()

# Also save static versions
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(study.trials) + 1), [t.value for t in study.trials], 'o-', markersize=3)
plt.xlabel('Trial')
plt.ylabel('ROC-AUC')
plt.title('Optimization History')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
importances = optuna.importance.get_param_importances(study)
plt.barh(list(importances.keys())[:10], list(importances.values())[:10])
plt.xlabel('Importance')
plt.title('Top 10 Parameter Importances')
plt.tight_layout()
plt.savefig('../docs/images/optuna_optimization.png', dpi=150)
plt.show()

print("Saved: docs/images/optuna_optimization.png")

## 9. Train Final XGBoost Model

In [ ]:
# Train final model with best parameters
print("Training final XGBoost model with best parameters...")

best_params = study.best_params.copy()
best_params.update({
    'random_state': 42,
    'n_jobs': -1,
    'eval_metric': 'logloss',
    'use_label_encoder': False,
})

xgb_final = XGBClassifier(**best_params)

start = time.time()
xgb_final.fit(X_train, y_train)
train_time = time.time() - start

print(f"Training completed in {train_time:.2f} seconds")

## 10. Evaluate XGBoost Model

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate model and print metrics."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n{'='*60}")
    print(f"  {model_name} - Test Results")
    print(f"{'='*60}")
    
    # Classification report
    print("\n--- Classification Report ---")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'],
                                  digits=4))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print("\n--- Confusion Matrix ---")
    print(f"                Predicted")
    print(f"               Legit  Fraud")
    print(f"Actual Legit   {tn:5d}  {fp:5d}")
    print(f"       Fraud   {fn:5d}  {tp:5d}")
    
    # Key metrics
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_proba)
    
    print(f"\n--- Key Metrics ---")
    print(f"  Fraud Recall (Catch Rate): {recall:.4%}  <- Primary metric")
    print(f"  Fraud Precision:           {precision:.4%}")
    print(f"  F1-Score:                  {f1:.4f}")
    print(f"  ROC-AUC:                   {roc_auc:.4f}")
    
    return y_pred, y_proba, cm

y_pred_xgb, y_proba_xgb, cm_xgb = evaluate_model(xgb_final, X_test, y_test, "XGBoost (Optuna Tuned)")

## 11. Compare with Baseline Models

In [ ]:
# Load baseline predictions
models_dir = Path('../models')
baseline_preds = joblib.load(models_dir / 'baseline_predictions.pkl')

# Calculate metrics for all models
models_compare = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost (Optuna)'],
    'ROC-AUC': [
        roc_auc_score(baseline_preds['y_test'], baseline_preds['logreg_proba']),
        roc_auc_score(baseline_preds['y_test'], baseline_preds['rf_proba']),
        roc_auc_score(y_test, y_proba_xgb)
    ],
    'Fraud Recall': [
        confusion_matrix(baseline_preds['y_test'], baseline_preds['logreg_pred'])[1,1] / 
        confusion_matrix(baseline_preds['y_test'], baseline_preds['logreg_pred'])[1,:].sum(),
        confusion_matrix(baseline_preds['y_test'], baseline_preds['rf_pred'])[1,1] /
        confusion_matrix(baseline_preds['y_test'], baseline_preds['rf_pred'])[1,:].sum(),
        cm_xgb[1,1] / cm_xgb[1,:].sum()
    ],
    'Fraud Precision': [
        confusion_matrix(baseline_preds['y_test'], baseline_preds['logreg_pred'])[1,1] /
        confusion_matrix(baseline_preds['y_test'], baseline_preds['logreg_pred'])[:,1].sum(),
        confusion_matrix(baseline_preds['y_test'], baseline_preds['rf_pred'])[1,1] /
        confusion_matrix(baseline_preds['y_test'], baseline_preds['rf_pred'])[:,1].sum(),
        cm_xgb[1,1] / cm_xgb[:,1].sum()
    ]
})

print("\n" + "="*70)
print("              MODEL COMPARISON")
print("="*70)
print(models_compare.to_string(index=False))
print("="*70)

# Check KPIs from project_guide
print("\n" + "="*70)
print("              KPI CHECK (project_guide targets)")
print("="*70)
xgb_metrics = models_compare[models_compare['Model'] == 'XGBoost (Optuna)'].iloc[0]
print(f"  Target ROC-AUC >= 0.95:  Actual = {xgb_metrics['ROC-AUC']:.4f}  {'✅ PASS' if xgb_metrics['ROC-AUC'] >= 0.95 else '❌ FAIL'}")
print(f"  Target Recall >= 0.90:  Actual = {xgb_metrics['Fraud Recall']:.4f}  {'✅ PASS' if xgb_metrics['Fraud Recall'] >= 0.90 else '❌ FAIL'}")
print(f"  Target Precision >= 0.85:  Actual = {xgb_metrics['Fraud Precision']:.4f}  {'✅ PASS' if xgb_metrics['Fraud Precision'] >= 0.85 else '❌ FAIL'}")
print("="*70)

## 12. ROC Curves Comparison

In [ ]:
plt.figure(figsize=(10, 8))

# Plot all models
models_data = {
    'Logistic Regression': baseline_preds['logreg_proba'],
    'Random Forest': baseline_preds['rf_proba'],
    'XGBoost (Optuna)': y_proba_xgb
}

for name, y_proba in models_data.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5000)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curves - All Models Comparison', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.tight_layout()
plt.savefig('../docs/images/all_models_roc_curves.png', dpi=150)
plt.show()

print("Saved: docs/images/all_models_roc_curves.png")

## 13. Save XGBoost Model

In [ ]:
# Save XGBoost model
models_dir = Path('../models')
joblib.dump(xgb_final, models_dir / 'xgboost_optuna.pkl')

# Save Optuna study for later analysis
joblib.dump(study, models_dir / 'optuna_study.pkl')

# Save predictions
predictions = {
    'y_test': y_test,
    'y_pred': y_pred_xgb,
    'y_proba': y_proba_xgb,
    'best_params': study.best_params,
    'best_score': study.best_value
}
joblib.dump(predictions, models_dir / 'xgboost_predictions.pkl')

print("\n=== Files Saved ===")
print(f"  {models_dir / 'xgboost_optuna.pkl'}")
print(f"  {models_dir / 'optuna_study.pkl'}")
print(f"  {models_dir / 'xgboost_predictions.pkl'}")

## 14. Summary Report

In [ ]:
print("\n" + "="*70)
print("              XGBOOST + OPTUNA - SUMMARY")
print("="*70)

print("\nModel Trained:")
print("  - XGBoost Classifier with Optuna hyperparameter tuning")
print(f"  - {len(study.trials)} trials completed")

print("\nBest ROC-AUC Score:")
print(f"  - Cross-validation: {study.best_value:.4f}")
print(f"  - Test set: {roc_auc_score(y_test, y_proba_xgb):.4f}")

print("\nKey Hyperparameters Found:")
for param in ['max_depth', 'learning_rate', 'n_estimators', 'subsample', 'scale_pos_weight']:
    if param in study.best_params:
        print(f"  - {param}: {study.best_params[param]}")

print("\nNext Steps:")
print("  - Day 6: Address class imbalance with advanced techniques")
print("  - Day 6: Compute full evaluation metrics")
print("  - Day 7: Select final model and create model card")

print("="*70)